In [0]:
-- ============================================================
-- NOTEBOOK: 02_silver_cleaning.sql
-- PURPOSE : Clean, deduplicate and conform Bronze → Silver.
--
-- CATALOG STRUCTURE:
--   {catalog}.bronze.raw_*  →  {catalog}.silver.*
--   catalog = yt_dev | uat | prod  (passed as job parameter)
--
-- AUTHOR  : The Data Master | Naval Yemul
-- ============================================================

-- Python cell at top of notebook:
--   env         = dbutils.widgets.get("env")
--   catalog     = env
--   spark.conf.set("pipeline.catalog", catalog)


-- ─────────────────────────────────────────────────────────────
-- SILVER 1 : customers
-- ─────────────────────────────────────────────────────────────
-- ① Dedup on customer_id  → keep latest by created_date
-- ② Null email            → 'unknown@nodomain.com'
-- ③ Null phone            → 'NOT_PROVIDED'
-- ④ Null annual_income_usd→ 0.0
-- ⑤ Trim / UPPER / INITCAP standardisation
-- ⑥ dq_flag + dq_notes   → track which fields were imputed

CREATE OR REPLACE TABLE IDENTIFIER(:catalog || '.silver.customers')
  USING DELTA
  TBLPROPERTIES (
    'delta.autoOptimize.optimizeWrite' = 'true',
    'delta.autoOptimize.autoCompact'   = 'true',
    'pipeline.layer'                   = 'silver',
    'pipeline.source_table'            = 'bronze.raw_customers',
    'pipeline.loaded_by'               = 'naval.yemul'
  )
  COMMENT 'Silver: cleansed customers — one row per customer_id.'
AS
WITH ranked AS (
  SELECT *,
    ROW_NUMBER() OVER (
      PARTITION BY customer_id
      ORDER BY created_date DESC, _ingested_at DESC
    ) AS _row_num
  FROM IDENTIFIER(:catalog || '.bronze.raw_customers')
  WHERE customer_id IS NOT NULL
),
deduped AS (
  SELECT * FROM ranked WHERE _row_num = 1
)
SELECT
  customer_id,
  TRIM(full_name)                                              AS full_name,
  LOWER(TRIM(COALESCE(email, 'unknown@nodomain.com')))        AS email,
  COALESCE(TRIM(phone), 'NOT_PROVIDED')                       AS phone,
  UPPER(TRIM(country))                                        AS country,
  INITCAP(account_type)                                       AS account_type,
  CAST(credit_score AS INT)                                   AS credit_score,
  CAST(COALESCE(annual_income_usd, 0.0) AS DECIMAL(18,2))    AS annual_income_usd,
  TO_DATE(created_date, 'yyyy-MM-dd')                         AS created_date,
  CAST(is_active AS BOOLEAN)                                  AS is_active,
  -- data quality tracking
  CASE
    WHEN email IS NULL OR phone IS NULL
      OR annual_income_usd IS NULL                            THEN TRUE
    ELSE FALSE
  END                                                         AS dq_flag,
  TRIM(
    CASE WHEN email            IS NULL THEN 'email | '            ELSE '' END ||
    CASE WHEN phone            IS NULL THEN 'phone | '            ELSE '' END ||
    CASE WHEN annual_income_usd IS NULL THEN 'annual_income_usd | ' ELSE '' END
  )                                                           AS dq_notes,
  _source_file,
  _ingested_at,
  current_timestamp()                                         AS _silver_loaded_at,
  :catalog                                                AS _catalog
FROM deduped;


-- ─────────────────────────────────────────────────────────────
-- SILVER 2 : accounts
-- ─────────────────────────────────────────────────────────────
-- ① Dedup on account_id → keep latest by open_date
-- ② Null balance        → 0.0
-- ③ Null credit_limit   → 0.0
-- ④ Null branch_code    → 'UNKNOWN'
-- ⑤ Null last_transaction_date → open_date
-- ⑥ Derived: available_credit = credit_limit − ABS(overdraft)
-- ⑦ Status → UPPER

CREATE OR REPLACE TABLE IDENTIFIER(:catalog || '.silver.accounts')
  USING DELTA
  TBLPROPERTIES (
    'delta.autoOptimize.optimizeWrite' = 'true',
    'delta.autoOptimize.autoCompact'   = 'true',
    'pipeline.layer'                   = 'silver',
    'pipeline.source_table'            = 'bronze.raw_accounts',
    'pipeline.loaded_by'               = 'naval.yemul'
  )
  COMMENT 'Silver: cleansed accounts — one row per account_id.'
AS
WITH ranked AS (
  SELECT *,
    ROW_NUMBER() OVER (
      PARTITION BY account_id
      ORDER BY open_date DESC, _ingested_at DESC
    ) AS _row_num
  FROM IDENTIFIER(:catalog || '.bronze.raw_accounts')
  WHERE account_id IS NOT NULL
),
deduped AS (SELECT * FROM ranked WHERE _row_num = 1)
SELECT
  account_id,
  customer_id,
  INITCAP(account_type)                                               AS account_type,
  UPPER(TRIM(currency))                                               AS currency,
  CAST(COALESCE(balance,      0.0) AS DECIMAL(18,2))                 AS balance,
  CAST(COALESCE(credit_limit, 0.0) AS DECIMAL(18,2))                 AS credit_limit,
  CAST(
    COALESCE(credit_limit, 0.0) - ABS(LEAST(COALESCE(balance, 0.0), 0.0))
  AS DECIMAL(18,2))                                                   AS available_credit,
  COALESCE(TRIM(branch_code), 'UNKNOWN')                             AS branch_code,
  TO_DATE(open_date, 'yyyy-MM-dd')                                    AS open_date,
  TO_DATE(COALESCE(last_transaction_date, open_date), 'yyyy-MM-dd')  AS last_transaction_date,
  UPPER(TRIM(status))                                                 AS status,
  CASE
    WHEN balance IS NULL OR branch_code IS NULL
      OR credit_limit IS NULL
      OR last_transaction_date IS NULL                                THEN TRUE
    ELSE FALSE
  END                                                                 AS dq_flag,
  _source_file,
  _ingested_at,
  current_timestamp()                                                 AS _silver_loaded_at,
  :catalog                                                        AS _catalog
FROM deduped;


-- ─────────────────────────────────────────────────────────────
-- SILVER 3 : transactions
-- ─────────────────────────────────────────────────────────────
-- ① Dedup on transaction_id → keep latest
-- ② Null amount             → 0.0  + dq_flag
-- ③ Null merchant_category  → 'UNCATEGORISED'
-- ④ Null merchant_name      → 'UNKNOWN_MERCHANT'
-- ⑤ Null channel            → 'UNKNOWN'
-- ⑥ Null description        → 'NO_DESCRIPTION'
-- ⑦ Combine date + time     → transaction_ts TIMESTAMP
-- ⑧ Derive year / month     → partition columns
-- ⑨ Standardise to UPPER throughout

CREATE OR REPLACE TABLE IDENTIFIER(:catalog || '.silver.transactions')
  USING DELTA
  PARTITIONED BY (transaction_year, transaction_month)
  TBLPROPERTIES (
    'delta.autoOptimize.optimizeWrite' = 'true',
    'delta.autoOptimize.autoCompact'   = 'true',
    'pipeline.layer'                   = 'silver',
    'pipeline.source_table'            = 'bronze.raw_transactions',
    'pipeline.loaded_by'               = 'naval.yemul'
  )
  COMMENT 'Silver: cleansed transactions partitioned by year/month.'
AS
WITH ranked AS (
  SELECT *,
    ROW_NUMBER() OVER (
      PARTITION BY transaction_id
      ORDER BY transaction_date DESC, _ingested_at DESC
    ) AS _row_num
  FROM IDENTIFIER(:catalog || '.bronze.raw_transactions')
  WHERE transaction_id IS NOT NULL
),
deduped AS (SELECT * FROM ranked WHERE _row_num = 1)
SELECT
  transaction_id,
  account_id,
  customer_id,
  TO_TIMESTAMP(
    CONCAT(transaction_date, ' ', COALESCE(transaction_time, '00:00:00')),
    'yyyy-MM-dd HH:mm:ss'
  )                                                               AS transaction_ts,
  TO_DATE(transaction_date, 'yyyy-MM-dd')                         AS transaction_date,
  YEAR(TO_DATE(transaction_date,  'yyyy-MM-dd'))                  AS transaction_year,
  MONTH(TO_DATE(transaction_date, 'yyyy-MM-dd'))                  AS transaction_month,
  CAST(COALESCE(amount, 0.0) AS DECIMAL(18,2))                   AS amount,
  UPPER(TRIM(currency))                                           AS currency,
  UPPER(TRIM(transaction_type))                                   AS transaction_type,
  COALESCE(UPPER(TRIM(merchant_category)), 'UNCATEGORISED')       AS merchant_category,
  COALESCE(TRIM(merchant_name),            'UNKNOWN_MERCHANT')    AS merchant_name,
  COALESCE(UPPER(TRIM(channel)),           'UNKNOWN')             AS channel,
  UPPER(TRIM(status))                                             AS status,
  reference_number,
  COALESCE(description, 'NO_DESCRIPTION')                         AS description,
  CAST(is_flagged AS BOOLEAN)                                     AS is_flagged,
  CASE
    WHEN amount IS NULL OR merchant_category IS NULL
      OR channel IS NULL                                          THEN TRUE
    ELSE FALSE
  END                                                             AS dq_flag,
  TRIM(
    CASE WHEN amount            IS NULL THEN 'amount | '            ELSE '' END ||
    CASE WHEN merchant_category IS NULL THEN 'merchant_category | ' ELSE '' END ||
    CASE WHEN channel           IS NULL THEN 'channel | '           ELSE '' END
  )                                                               AS dq_notes,
  _source_file,
  _ingested_at,
  current_timestamp()                                             AS _silver_loaded_at,
  :catalog                                                    AS _catalog
FROM deduped;


-- ─────────────────────────────────────────────────────────────
-- SILVER 4 : loans
-- ─────────────────────────────────────────────────────────────
-- ① Dedup on loan_id → keep latest by disbursement_date
-- ② Null outstanding_balance → 0.0
-- ③ Null collateral_type     → 'NONE'
-- ④ Null overdue_days        → 0
-- ⑤ Derived: repayment_pct, loan_health bucket

CREATE OR REPLACE TABLE IDENTIFIER(:catalog || '.silver.loans')
  USING DELTA
  TBLPROPERTIES (
    'delta.autoOptimize.optimizeWrite' = 'true',
    'delta.autoOptimize.autoCompact'   = 'true',
    'pipeline.layer'                   = 'silver',
    'pipeline.source_table'            = 'bronze.raw_loans',
    'pipeline.loaded_by'               = 'naval.yemul'
  )
  COMMENT 'Silver: cleansed loans with derived loan_health bucket.'
AS
WITH ranked AS (
  SELECT *,
    ROW_NUMBER() OVER (
      PARTITION BY loan_id
      ORDER BY disbursement_date DESC, _ingested_at DESC
    ) AS _row_num
  FROM IDENTIFIER(:catalog || '.bronze.raw_loans')
  WHERE loan_id IS NOT NULL
),
deduped AS (SELECT * FROM ranked WHERE _row_num = 1)
SELECT
  loan_id,
  customer_id,
  account_id,
  INITCAP(loan_type)                                               AS loan_type,
  CAST(principal_amount                   AS DECIMAL(18,2))        AS principal_amount,
  CAST(COALESCE(outstanding_balance, 0.0) AS DECIMAL(18,2))        AS outstanding_balance,
  ROUND(
    (1 - COALESCE(outstanding_balance, 0.0) / NULLIF(principal_amount, 0)) * 100, 2
  )                                                                AS repayment_pct,
  CAST(interest_rate_pct                  AS DECIMAL(6,3))         AS interest_rate_pct,
  CAST(tenure_months                      AS INT)                  AS tenure_months,
  CAST(emi_amount                         AS DECIMAL(18,2))        AS emi_amount,
  TO_DATE(disbursement_date, 'yyyy-MM-dd')                         AS disbursement_date,
  TO_DATE(next_due_date,     'yyyy-MM-dd')                         AS next_due_date,
  CAST(COALESCE(overdue_days, 0)          AS INT)                  AS overdue_days,
  CASE
    WHEN UPPER(status) = 'DEFAULTED'                  THEN 'DEFAULTED'
    WHEN COALESCE(overdue_days, 0) > 90               THEN 'DEFAULTED'
    WHEN COALESCE(overdue_days, 0) BETWEEN 31 AND 90  THEN 'AT_RISK'
    WHEN COALESCE(overdue_days, 0) BETWEEN 1  AND 30  THEN 'WATCHLIST'
    ELSE                                                   'GOOD'
  END                                                              AS loan_health,
  UPPER(TRIM(status))                                              AS status,
  COALESCE(UPPER(TRIM(collateral_type)), 'NONE')                  AS collateral_type,
  CASE
    WHEN outstanding_balance IS NULL
      OR collateral_type IS NULL
      OR overdue_days IS NULL                                      THEN TRUE
    ELSE FALSE
  END                                                              AS dq_flag,
  _source_file,
  _ingested_at,
  current_timestamp()                                              AS _silver_loaded_at,
  :catalog                                                     AS _catalog
FROM deduped;


-- ─────────────────────────────────────────────────────────────
-- SILVER VALIDATION
-- ─────────────────────────────────────────────────────────────

SELECT 'silver.customers'     AS table_name,
       COUNT(*)               AS total_rows,
       SUM(CASE WHEN dq_flag THEN 1 ELSE 0 END) AS rows_flagged
FROM IDENTIFIER(:catalog || '.silver.customers')
UNION ALL
SELECT 'silver.accounts',     COUNT(*), SUM(CASE WHEN dq_flag THEN 1 ELSE 0 END)
FROM IDENTIFIER(:catalog || '.silver.accounts')
UNION ALL
SELECT 'silver.transactions', COUNT(*), SUM(CASE WHEN dq_flag THEN 1 ELSE 0 END)
FROM IDENTIFIER(:catalog || '.silver.transactions')
UNION ALL
SELECT 'silver.loans',        COUNT(*), SUM(CASE WHEN dq_flag THEN 1 ELSE 0 END)
FROM IDENTIFIER(:catalog || '.silver.loans');